# BRIO training: 32 candidates, 6,000 steps, MLE weight 1, ranking weight (γ) 100

Trains BRIO (IndoBART-v2, Liputan6 canonical) on the 96,000-article subset with 32 candidates per article (`liputan6_32_6000.zip`).

- **Loss:** `1 × MLE + 100 × ranking`. The other two notebooks (`BRIO_32c_6000_1-1`, `1-10`, `1-100`) differ only in `RANK_WEIGHT`.
- **Length:** 96,000 articles ÷ 16 per step (batch size 2 × accumulate 8) = 6,000 steps, 1 epoch. Validation on the full canonical val set at steps 2,000, 4,000 and 6,000; the best by validation is saved as `model_generation.bin`.
- **GPU:** training runs in bfloat16 (`main.py`), so use a runtime that supports it: **L4 or A100**. A T4 has no bfloat16 hardware and would be much slower.
- **Checkpoints:** every 2,000 steps to Drive, in `CKPT_DIR/<step>/` (weights, optimizer, training state and, from now on, `log.txt`). After a disconnect, open the notebook again and run all cells: it resumes from the latest step on Drive. Resuming first skips the batches already trained, which takes a while.
- **Never** run the same notebook in two sessions at the same time: both would write to the same Drive folder.

In [1]:
# Install condacolab. The runtime restarts after this cell, that is normal.
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to:

    !pip install -q 'condacolab==0.1.*'

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:07
🔁 Restarting kernel...


In [1]:
# Clone the repo and mount Google Drive.
!git clone https://github.com/alvrian/BRIO.git

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%cd BRIO

Cloning into 'BRIO'...
remote: Enumerating objects: 616, done.
remote: Counting objects: 100% (325/325), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 616 (delta 232), reused 159 (delta 105), pack-reused 291 (from 2)
Receiving objects: 100% (616/616), 16.18 MiB | 15.99 MiB/s, done.
Resolving deltas: 100% (386/386), done.
Mounted at /content/drive
/content/BRIO


In [2]:
# Env setup (same as the other notebooks) and the fine-tuned IndoBART that BRIO starts from.
! conda create --name env --file spec-file.txt -y
! conda run --no-capture-output -n env pip install -r requirements.txt
! conda run -n env python -c "import nltk;nltk.download('punkt')"
! git clone https://github.com/neulab/compare-mt.git
! cd compare-mt && conda run --no-capture-output -n env pip install -r requirements.txt
! cd compare-mt && conda run -n env python setup.py install
print("done setting up env")
! cp /content/drive/MyDrive/BRIO/indobart-liputan6-finetuned.zip .
! unzip -q indobart-liputan6-finetuned.zip
! rm indobart-liputan6-finetuned.zip
! mkdir -p cache result
print("all done")

Retrieving notices: - done

_libgcc_mutex-0.1    | :   0% 0/1 [00:00<?, ?it/s]
blas-1.0             | :   0% 0/1 [00:00<?, ?it/s]

ca-certificates-2021 | :   0% 0/1 [00:00<?, ?it/s]


intel-openmp-2021.3. | :   0% 0/1 [00:00<?, ?it/s]



ld_impl_linux-64-2.3 | :   0% 0/1 [00:00<?, ?it/s]




libstdcxx-ng-9.3.0   | :   0% 0/1 [00:00<?, ?it/s]





libgomp-9.3.0        | :   0% 0/1 [00:00<?, ?it/s]






mkl-2021.3.0         | :   0% 0/1 [00:00<?, ?it/s]







_openmp_mutex-4.5    | :   0% 0/1 [00:00<?, ?it/s]








libgcc-ng-9.3.0      | :   0% 0/1 [00:00<?, ?it/s]









bzip2-1.0.8          | :   0% 0/1 [00:00<?, ?it/s]










cudatoolkit-11.1.1   | :   0% 0/1 [00:00<?, ?it/s]











giflib-5.2.1         | :   0% 0/1 [00:00<?, ?it/s]












gmp-6.2.1            | :   0% 0/1 [00:00<?, ?it/s]













jpeg-9b              | :   0% 0/1 [00:00<?, ?it/s]














lame-3.100           | :   0% 0/1 [00:00<?, ?it/s]















libffi-3.3           | :   0% 0/1 

In [3]:
# Settings
import os, glob, json, shutil

MLE_WEIGHT = 1
RANK_WEIGHT = 100            # gamma
SEED = 970903                # config.py default; change for further seeds
NUM_CANDS = 32
BATCH_SIZE, ACCUM = 1, 16     # 16 articles per step. Out of GPU memory: use 1, 16 (same 16 per step, same 6,000 steps)
TOTAL_STEPS = 6000
# assert BATCH_SIZE * ACCUM == 16

# the dataset zip: MyDrive/BRIO/Dataset/ (or dataset/)
candidates = ["/content/drive/MyDrive/BRIO/Dataset/liputan6_32_6000.zip", "/content/drive/MyDrive/BRIO/dataset/liputan6_32_6000.zip"]
DATASET_ZIP = next((p for p in candidates if os.path.exists(p)), None)
print("dataset zip:", DATASET_ZIP or "NOT FOUND in " + " or ".join(candidates))

CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_%s-%s_seed%d" % (MLE_WEIGHT, RANK_WEIGHT, SEED)
print("checkpoints:", CKPT_DIR)
print("model:", "found" if os.path.exists("./indobart-liputan6-finetuned/config.json") else "MISSING", "./indobart-liputan6-finetuned")

dataset zip: /content/drive/MyDrive/BRIO/Dataset/liputan6_32_6000.zip
checkpoints: /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_1-100_seed970903
model: found ./indobart-liputan6-finetuned


In [4]:
# Restore the preprocessed dataset into ./liputan6 and check it.
import zipfile, random
if not os.path.exists("./liputan6/canonical/train"):
    shutil.copyfile(DATASET_ZIP, "./liputan6_32_6000.zip")
    with zipfile.ZipFile("./liputan6_32_6000.zip") as z:
        z.extractall("./liputan6")
    os.remove("./liputan6_32_6000.zip")

expected = {"train": 96000, "val": 10972, "test": 10972}
for split, n in expected.items():
    found = len(os.listdir("./liputan6/canonical/" + split))
    print("%-5s %6d json files %s" % (split, found, "ok" if found == n else "EXPECTED %d" % n))
for p in ["./liputan6/canonical/test.source", "./liputan6/canonical/test.target", "./liputan6/test.target"]:
    print("found  " if os.path.exists(p) else "MISSING", p)
with open("./liputan6/canonical/train/%d.json" % random.randrange(expected["train"])) as f:
    print("candidates per article:", len(json.load(f)["candidates"]))
print("steps in 1 epoch: %d" % (expected["train"] // (BATCH_SIZE * ACCUM)))

train  96000 json files ok
val    10972 json files ok
test   10972 json files ok
found   ./liputan6/canonical/test.source
found   ./liputan6/canonical/test.target
found   ./liputan6/test.target
candidates per article: 32
steps in 1 epoch: 6000


In [5]:
# GPU check: training uses bfloat16.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!conda run -n env python -c "import torch; c = torch.cuda.get_device_capability(); print('compute capability', c, '- bfloat16 in hardware' if c[0] >= 8 else '- NO bfloat16 hardware (T4?): switch to an L4 or A100 runtime')"

NVIDIA L4, 23034 MiB
compute capability (8, 9) - bfloat16 in hardware


In [6]:
# Resume: find the latest step saved on Drive and copy it into ./cache, so main.py continues from there.
def saved_steps():
    steps = []
    for d in glob.glob(os.path.join(CKPT_DIR, "*")):
        name = os.path.basename(d)
        if name.isdigit() and all(os.path.exists(os.path.join(d, f)) for f in ["model_cur.bin", "optimizer.bin", "train_state.json"]):
            steps.append(int(name))
    return sorted(steps)

steps = saved_steps()
MODEL_PT_FLAG = ""
TRAIN = True
if not steps:
    print("no checkpoint on Drive yet: training starts from step 0")
elif steps[-1] >= TOTAL_STEPS:
    TRAIN = False
    print("step %d is already on Drive: training is finished, skip the next cell" % steps[-1])
else:
    last = steps[-1]
    resume_dir = "./cache/resume_step%d" % last
    os.makedirs(resume_dir, exist_ok=True)
    for f in ["model_cur.bin", "optimizer.bin", "train_state.json"]:
        shutil.copyfile(os.path.join(CKPT_DIR, str(last), f), os.path.join(resume_dir, f))
    with open(os.path.join(resume_dir, "train_state.json")) as f:
        print("resuming from", json.load(f))
    MODEL_PT_FLAG = "--model_pt resume_step%d/model_cur.bin" % last
print("saved steps:", steps)

step 6000 is already on Drive: training is finished, skip the next cell
saved steps: [4000, 6000]


In [ ]:
# Train. Rerun all cells after a disconnect: it resumes from the latest step on Drive.
if TRAIN:
    !conda run --no-capture-output -n env python main.py --cuda --gpuid 0 --config liputan6 -l --seed {SEED} --max_num {NUM_CANDS} --batch_size {BATCH_SIZE} --accumulate_step {ACCUM} --mle_weight {MLE_WEIGHT} --rank_weight {RANK_WEIGHT} --checkpoint_save_dir "{CKPT_DIR}" {MODEL_PT_FLAG}
else:
    print("training already finished")

start...




start processing data in ./liputan6/canonical/train
start processing data in ./liputan6/canonical/val
done extracting data for liputan6 for BRIO
start building model
Namespace(accumulate_step=16, adding=0, batch_size=1, checkpoint_save_dir='/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_1-100_seed970903', config='liputan6', cuda=True, dataset='liputan6', dataset_folder=None, datatype='canonical', do_generation=False, do_reranking=False, do_sample=True, epoch=1, eval_interval=2000, evaluate=False, gen_max_len=100, gen_min_len=20, gold_margin=0, gold_weight=0, gpuid=[0], grad_norm=0, is_pegasus=False, length_penalty=1.0, log=True, margin=0.001, max_len=100, max_lr=0.0007, max_num=32, mle_weight=1, model_pt='', model_type='indobenchmark/indobart-v2', no_gold=False, normalize=True, num_beams=4, port=12355, pretrained='indobart-liputan6-finetuned', rank_weight=100, report_freq=200, scale=1, score_mode='log', seed=970903, smooth=0.1, total_len=1024, warmup_steps=

In [ ]:
# Seed 2
SEED = 70905
CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_%s-%s_seed%d" % (MLE_WEIGHT, RANK_WEIGHT, SEED)
steps = saved_steps()
MODEL_PT_FLAG = ""
if steps and steps[-1] >= TOTAL_STEPS:
    print("seed %d: step %d already on Drive, training finished" % (SEED, steps[-1]))
else:
    if steps:
        resume_dir = "./cache/resume_seed%d_step%d" % (SEED, steps[-1])
        os.makedirs(resume_dir, exist_ok=True)
        for f in ["model_cur.bin", "optimizer.bin", "train_state.json"]:
            shutil.copyfile(os.path.join(CKPT_DIR, str(steps[-1]), f), os.path.join(resume_dir, f))
        MODEL_PT_FLAG = "--model_pt resume_seed%d_step%d/model_cur.bin" % (SEED, steps[-1])
        print("seed %d: resuming from step %d" % (SEED, steps[-1]))
    else:
        print("seed %d: starting from step 0" % SEED)
    print("checkpoints:", CKPT_DIR)
    !conda run --no-capture-output -n env python main.py --cuda --gpuid 0 --config liputan6 -l --seed {SEED} --max_num {NUM_CANDS} --batch_size {BATCH_SIZE} --accumulate_step {ACCUM} --mle_weight {MLE_WEIGHT} --rank_weight {RANK_WEIGHT} --checkpoint_save_dir "{CKPT_DIR}" {MODEL_PT_FLAG}

seed 70905: starting from step 0
checkpoints: /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_1-100_seed70905
start...
start processing data in ./liputan6/canonical/train
start processing data in ./liputan6/canonical/val
done extracting data for liputan6 for BRIO
start building model
Namespace(accumulate_step=16, adding=0, batch_size=1, checkpoint_save_dir='/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_1-100_seed70905', config='liputan6', cuda=True, dataset='liputan6', dataset_folder=None, datatype='canonical', do_generation=False, do_reranking=False, do_sample=True, epoch=1, eval_interval=2000, evaluate=False, gen_max_len=100, gen_min_len=20, gold_margin=0, gold_weight=0, gpuid=[0], grad_norm=0, is_pegasus=False, length_penalty=1.0, log=True, margin=0.001, max_len=100, max_lr=0.0007, max_num=32, mle_weight=1, model_pt='', model_type='indobenchmark/indobart-v2', no_gold=False, normalize=True, num_beams=4, port=12355, pretrained='indobart-liputan6-finetun

In [ ]:
# Seed 3
SEED = 81215
CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_%s-%s_seed%d" % (MLE_WEIGHT, RANK_WEIGHT, SEED)
steps = saved_steps()
MODEL_PT_FLAG = ""
if steps and steps[-1] >= TOTAL_STEPS:
    print("seed %d: step %d already on Drive, training finished" % (SEED, steps[-1]))
else:
    if steps:
        resume_dir = "./cache/resume_seed%d_step%d" % (SEED, steps[-1])
        os.makedirs(resume_dir, exist_ok=True)
        for f in ["model_cur.bin", "optimizer.bin", "train_state.json"]:
            shutil.copyfile(os.path.join(CKPT_DIR, str(steps[-1]), f), os.path.join(resume_dir, f))
        MODEL_PT_FLAG = "--model_pt resume_seed%d_step%d/model_cur.bin" % (SEED, steps[-1])
        print("seed %d: resuming from step %d" % (SEED, steps[-1]))
    else:
        print("seed %d: starting from step 0" % SEED)
    print("checkpoints:", CKPT_DIR)
    !conda run --no-capture-output -n env python main.py --cuda --gpuid 0 --config liputan6 -l --seed {SEED} --max_num {NUM_CANDS} --batch_size {BATCH_SIZE} --accumulate_step {ACCUM} --mle_weight {MLE_WEIGHT} --rank_weight {RANK_WEIGHT} --checkpoint_save_dir "{CKPT_DIR}" {MODEL_PT_FLAG}

seed 81215: starting from step 0
checkpoints: /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_1-100_seed81215
start...
start processing data in ./liputan6/canonical/train
start processing data in ./liputan6/canonical/val
done extracting data for liputan6 for BRIO
start building model
Namespace(accumulate_step=16, adding=0, batch_size=1, checkpoint_save_dir='/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_1-100_seed81215', config='liputan6', cuda=True, dataset='liputan6', dataset_folder=None, datatype='canonical', do_generation=False, do_reranking=False, do_sample=True, epoch=1, eval_interval=2000, evaluate=False, gen_max_len=100, gen_min_len=20, gold_margin=0, gold_weight=0, gpuid=[0], grad_norm=0, is_pegasus=False, length_penalty=1.0, log=True, margin=0.001, max_len=100, max_lr=0.0007, max_num=32, mle_weight=1, model_pt='', model_type='indobenchmark/indobart-v2', no_gold=False, normalize=True, num_beams=4, port=12355, pretrained='indobart-liputan6-finetun

In [7]:
# Train any gamma / seed in this session, one after another. Resumes from Drive like the other cells.
def train(rank_weight, seed):
    global SEED, RANK_WEIGHT, CKPT_DIR
    SEED, RANK_WEIGHT = seed, rank_weight
    CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_%s-%s_seed%d" % (MLE_WEIGHT, RANK_WEIGHT, SEED)
    steps = saved_steps()
    model_pt = ""
    if steps and steps[-1] >= TOTAL_STEPS:
        print("gamma %s seed %d: step %d already on Drive, training finished" % (RANK_WEIGHT, SEED, steps[-1]))
        return
    if steps:
        resume_dir = "./cache/resume_%s_seed%d_step%d" % (RANK_WEIGHT, SEED, steps[-1])
        os.makedirs(resume_dir, exist_ok=True)
        for f in ["model_cur.bin", "optimizer.bin", "train_state.json"]:
            shutil.copyfile(os.path.join(CKPT_DIR, str(steps[-1]), f), os.path.join(resume_dir, f))
        model_pt = "--model_pt %s/model_cur.bin" % os.path.basename(resume_dir)
        print("gamma %s seed %d: resuming from step %d" % (RANK_WEIGHT, SEED, steps[-1]))
    else:
        print("gamma %s seed %d: starting from step 0" % (RANK_WEIGHT, SEED))
    print("checkpoints:", CKPT_DIR)
    !conda run --no-capture-output -n env python main.py --cuda --gpuid 0 --config liputan6 -l --seed {SEED} --max_num {NUM_CANDS} --batch_size {BATCH_SIZE} --accumulate_step {ACCUM} --mle_weight {MLE_WEIGHT} --rank_weight {RANK_WEIGHT} --checkpoint_save_dir "{CKPT_DIR}" {model_pt}

In [ ]:
train(10, 970903)

gamma 10 seed 970903: starting from step 0
checkpoints: /content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_1-10_seed970903
start...




start processing data in ./liputan6/canonical/train
start processing data in ./liputan6/canonical/val
done extracting data for liputan6 for BRIO
start building model
Namespace(accumulate_step=16, adding=0, batch_size=1, checkpoint_save_dir='/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_32c_6000_1-10_seed970903', config='liputan6', cuda=True, dataset='liputan6', dataset_folder=None, datatype='canonical', do_generation=False, do_reranking=False, do_sample=True, epoch=1, eval_interval=2000, evaluate=False, gen_max_len=100, gen_min_len=20, gold_margin=0, gold_weight=0, gpuid=[0], grad_norm=0, is_pegasus=False, length_penalty=1.0, log=True, margin=0.001, max_len=100, max_lr=0.0007, max_num=32, mle_weight=1, model_pt='', model_type='indobenchmark/indobart-v2', no_gold=False, normalize=True, num_beams=4, port=12355, pretrained='indobart-li

In [ ]:
train(10, 70905)

In [ ]:
train(10, 81215)

In [ ]:
# 8-candidate dataset into ./liputan6_8/canonical (the 32-candidate one stays in ./liputan6)
import zipfile, glob, shutil, os, json
DATA8 = "./liputan6_8"
zips = ["/content/drive/MyDrive/BRIO/dataset/liputan6_8.zip", "/content/drive/MyDrive/BRIO/Dataset/liputan6_8.zip"]
ZIP8 = next((p for p in zips if os.path.exists(p)), None)
assert ZIP8, "liputan6_8.zip not found in " + " or ".join(zips)
if not os.path.exists(os.path.join(DATA8, "canonical", "train")):
    shutil.copyfile(ZIP8, "./liputan6_8.zip")
    with zipfile.ZipFile("./liputan6_8.zip") as z:
        z.extractall("./_unzip8")
    os.remove("./liputan6_8.zip")
    train_dirs = [d for d in glob.glob("./_unzip8/**/train", recursive=True) if os.path.isdir(d)]
    assert train_dirs, "no train folder found in the zip"
    os.makedirs(DATA8, exist_ok=True)
    shutil.move(os.path.dirname(train_dirs[0]), os.path.join(DATA8, "canonical"))
    shutil.rmtree("./_unzip8")
for split in ["train", "val", "test"]:
    print(split, len(os.listdir(os.path.join(DATA8, "canonical", split))), "json files")
with open(os.path.join(DATA8, "canonical", "train", "0.json")) as f:
    print("candidates per article:", len(json.load(f)["candidates"]))

In [ ]:
# BRIO gamma = 100 (MLE 1, ranking 100), 8 candidates, second seed - full train split
import re
CKPT_8 = "/content/drive/MyDrive/BRIO/checkpoints/BRIO_Liputan6_checkpoint_1-100_8_second_seed"

seeds = set()
for cfg in glob.glob("/content/drive/MyDrive/BRIO/checkpoints/*second_seed*/*/config.txt"):
    m = re.search(r"seed=(\d+)", open(cfg, encoding="utf-8").read())
    if m:
        seeds.add(int(m.group(1)))
        print("found seed", m.group(1), "in", cfg)
SEED_8 = seeds.pop() if len(seeds) == 1 else 70905
print("using seed", SEED_8, "" if len(seeds) == 0 else "- WARNING: several seeds found, check which one is right")

done = os.path.exists(os.path.join(CKPT_8, "master_1_100", "model_generation.bin"))
steps = sorted(int(os.path.basename(d)) for d in glob.glob(os.path.join(CKPT_8, "*"))
               if os.path.basename(d).isdigit() and
               all(os.path.exists(os.path.join(d, f)) for f in ["model_cur.bin", "optimizer.bin", "train_state.json"]))
pt_flag = ""
if done:
    print("training already finished:", os.path.join(CKPT_8, "master_1_100"))
else:
    if steps:
        rdir = "./cache/resume_8c_step%d" % steps[-1]
        os.makedirs(rdir, exist_ok=True)
        for f in ["model_cur.bin", "optimizer.bin", "train_state.json"]:
            shutil.copyfile(os.path.join(CKPT_8, str(steps[-1]), f), os.path.join(rdir, f))
        pt_flag = "--model_pt resume_8c_step%d/model_cur.bin" % steps[-1]
        print("resuming from step", steps[-1])
    !conda run --no-capture-output -n env python main.py --cuda --gpuid 0 --config liputan6 -l --seed {SEED_8} --max_num 8 --mle_weight 1 --rank_weight 100 --dataset_folder "{DATA8}" --checkpoint_save_dir "{CKPT_8}" {pt_flag}

## After training

The cell below lists the validation results of every saved step and finds the best model by validation (`model_generation.bin`). Each step folder holds the best model **up to that step** only when it improved in the session that saved it, so the latest step folder that has `model_generation.bin` holds the overall best.

Use that path for the evaluation notebooks (`generate_outputs.py`, `BRIO_bertscore.ipynb`, `BRIO_llm_judge.ipynb`), and add `--max_num 32` to `main.py -e`.

In [ ]:
import re
print("validation per saved step (from the log.txt synced with each step):")
for step in saved_steps():
    log = os.path.join(CKPT_DIR, str(step), "log.txt")
    if not os.path.exists(log):
        print("  step %5d: no log.txt (saved before log syncing was added)" % step)
        continue
    text = open(log, encoding="utf-8").read()
    gen = re.findall(r"val generation rouge1: ([\d.]+), rouge2: ([\d.]+), rougeLsum: ([\d.]+)", text)
    if gen:
        print("  step %5d: generation R1 %s  R2 %s  RLsum %s" % ((step,) + gen[-1]))

with_best = [s for s in saved_steps() if os.path.exists(os.path.join(CKPT_DIR, str(s), "model_generation.bin"))]
if with_best:
    print("\nbest model by validation:", os.path.join(CKPT_DIR, str(with_best[-1]), "model_generation.bin"))
master = glob.glob(os.path.join(CKPT_DIR, "master_*"))
print("end-of-epoch copy:", master[0] if master else "not there (training not finished)")